# Question 3: Full Stable Diffusion Inpainting Pipeline for Dog Image Completion

After the demo, we ran a **full Stable Diffusion inpainting workflow**. Again, we did **not train Stable Diffusion from scratch** as a full diffusion training would require a very large dataset and GPU resources. Instead, this part uses a **pretrained Stable Diffusion inpainting model** as the advanced generative model and applies it systematically to the same masked dog images used by the U-Net and edge-guided models.

The goal is to make this model comparable with the earlier notebooks:

1. `01_UNet` — plain U-Net baseline
2. `02_edge_generator` — edge completion model
3. `03_image_completion` — edge-guided U-Net image completion
4. `04_full_stable_diffusion` — pretrained diffusion-based inpainting model

## 1. Install and import packages

This section installs the libraries needed for Stable Diffusion inpainting. The key package is `diffusers`, which provides the pretrained inpainting pipeline.

In [ ]:
!pip install diffusers transformers accelerate safetensors scikit-image -q

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
from diffusers import StableDiffusionInpaintPipeline

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

## 2. Project paths and settings


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
BASE_DIR = Path('/content/drive/MyDrive/Adv CV/CV Project Folder')
DATA_DIR = BASE_DIR / "Images"
RESULT_DIR = BASE_DIR / 'results'
SD_RESULT_DIR = RESULT_DIR / 'stable_diffusion_full'
SD_IMAGE_DIR = SD_RESULT_DIR / 'images'
SD_FIGURE_DIR = SD_RESULT_DIR / 'figures'
SD_METRIC_DIR = SD_RESULT_DIR / 'metrics'

for d in [SD_RESULT_DIR, SD_IMAGE_DIR, SD_FIGURE_DIR, SD_METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 3. Load the pretrained Stable Diffusion inpainting model

This model is used as an advanced pretrained generative inpainting method. It has already learned strong natural-image priors, which is why it can produce sharper and more realistic completions than a small U-Net trained from scratch.

In [ ]:
MODEL_ID = 'runwayml/stable-diffusion-inpainting'

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to(DEVICE)

if DEVICE == 'cuda':
    pipe.enable_attention_slicing()
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print('xformers enabled')
    except Exception as e:
        print('xformers not available, continuing without it:', e)

print('Stable Diffusion inpainting pipeline loaded.')

## 4. Helper functions

These functions standardize image loading, mask creation, inpainting, visualization, and metric calculation.

Mask convention for Stable Diffusion:

- **White** pixels = area to inpaint
- **Black** pixels = area to preserve

In [ ]:
def center_square_crop(img):
    # Center-crop an image to a square before resizing.
    w, h = img.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    return img.crop((left, top, left + side, top + side))


def load_rgb_image(path, size=512):
    img = Image.open(path).convert('RGB')
    img = center_square_crop(img)
    img = img.resize((size, size), Image.BICUBIC)
    return img


def create_box_mask(size=512, box_ratio=0.32, jitter=True, seed=None):
    # Create a rectangular white inpainting mask on a black background.
    if seed is not None:
        random.seed(seed)
    mask = Image.new('L', (size, size), 0)
    box_w = int(size * box_ratio)
    box_h = int(size * box_ratio)
    if jitter:
        cx = random.randint(int(size * 0.35), int(size * 0.65))
        cy = random.randint(int(size * 0.35), int(size * 0.65))
    else:
        cx, cy = size // 2, size // 2
    left = max(0, cx - box_w // 2)
    top = max(0, cy - box_h // 2)
    right = min(size, left + box_w)
    bottom = min(size, top + box_h)
    arr = np.array(mask)
    arr[top:bottom, left:right] = 255
    return Image.fromarray(arr)


def apply_mask_to_image(image, mask):
    # Create a masked input image by blacking out the white mask region.
    img_arr = np.array(image).copy()
    mask_arr = np.array(mask)
    img_arr[mask_arr > 127] = 0
    return Image.fromarray(img_arr)


def run_sd_inpainting(image, mask, prompt, negative_prompt=None, seed=42,
                      guidance_scale=7.5, num_inference_steps=35, strength=1.0):
    generator = torch.Generator(device=DEVICE).manual_seed(seed) if DEVICE == 'cuda' else torch.Generator().manual_seed(seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        strength=strength,
        generator=generator
    ).images[0]
    return result


def compute_metrics(gt, pred, mask=None):
    # Compute L1, MSE, PSNR, and SSIM. Also compute masked-region metrics if mask is provided.
    gt_arr = np.array(gt).astype(np.float32) / 255.0
    pred_arr = np.array(pred).astype(np.float32) / 255.0

    metrics = {
        'l1_full': float(np.mean(np.abs(gt_arr - pred_arr))),
        'mse_full': float(np.mean((gt_arr - pred_arr) ** 2)),
        'psnr_full': float(psnr(gt_arr, pred_arr, data_range=1.0)),
        'ssim_full': float(ssim(gt_arr, pred_arr, channel_axis=2, data_range=1.0)),
    }

    if mask is not None:
        mask_arr = (np.array(mask).astype(np.float32) / 255.0) > 0.5
        if mask_arr.sum() > 0:
            diff_abs = np.abs(gt_arr - pred_arr)
            diff_sq = (gt_arr - pred_arr) ** 2
            metrics['l1_masked_region'] = float(diff_abs[mask_arr].mean())
            metrics['mse_masked_region'] = float(diff_sq[mask_arr].mean())
    return metrics


def show_comparison(gt, masked, mask, result, title=None, save_path=None):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    images = [gt, masked, mask, result]
    titles = ['Ground Truth', 'Masked Input', 'Mask', 'Stable Diffusion Completion']
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im, cmap='gray' if t == 'Mask' else None)
        ax.set_title(t)
        ax.axis('off')
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print('Saved figure to:', save_path)
    plt.show()

## 5. Select images for the full model run

This cell searches for dog images under `DATA_DIR`.

In [ ]:
image_paths = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    image_paths.extend(DATA_DIR.rglob(ext))

image_paths = sorted(image_paths)
print('Number of images found:', len(image_paths))
for p in image_paths[:5]:
    print(p)

## 6. Single-image test

Always run one image first before batch processing. This confirms that the model, mask, prompt, and output folder are working correctly.

In [ ]:
idx = 0
image_path = image_paths[idx]

SIZE = 512
BOX_RATIO = 0.32
SEED = 123

prompt = 'a realistic photo of a dog, natural fur texture, realistic lighting, high quality'
negative_prompt = 'blurry, distorted, deformed, extra limbs, cartoon, painting, low quality, artifacts'

gt = load_rgb_image(image_path, size=SIZE)
mask = create_box_mask(size=SIZE, box_ratio=BOX_RATIO, jitter=True, seed=SEED)
masked = apply_mask_to_image(gt, mask)

result = run_sd_inpainting(
    image=masked,
    mask=mask,
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=SEED,
    guidance_scale=7.5,
    num_inference_steps=35
)

metrics = compute_metrics(gt, result, mask)
print(metrics)

show_comparison(
    gt, masked, mask, result,
    title='Single-image Stable Diffusion Inpainting Demo',
    save_path=SD_FIGURE_DIR / 'single_image_sd_demo.png'
)

## 7. Batch inference on multiple images

This section applies Stable Diffusion inpainting to multiple dog images using the same setup. The output images and metrics are saved for later comparison against the U-Net and edge-guided models.


In [ ]:
NUM_IMAGES = min(20, len(image_paths))
BOX_RATIO = 0.32
SIZE = 512
BASE_SEED = 2026

records = []

for i, image_path in enumerate(image_paths[:NUM_IMAGES]):
    seed = BASE_SEED + i
    gt = load_rgb_image(image_path, size=SIZE)
    mask = create_box_mask(size=SIZE, box_ratio=BOX_RATIO, jitter=True, seed=seed)
    masked = apply_mask_to_image(gt, mask)

    result = run_sd_inpainting(
        image=masked,
        mask=mask,
        prompt=prompt,
        negative_prompt=negative_prompt,
        seed=seed,
        guidance_scale=7.5,
        num_inference_steps=35
    )

    stem = f'sd_{i:03d}'
    gt_path = SD_IMAGE_DIR / f'{stem}_ground_truth.png'
    masked_path = SD_IMAGE_DIR / f'{stem}_masked_input.png'
    mask_path = SD_IMAGE_DIR / f'{stem}_mask.png'
    result_path = SD_IMAGE_DIR / f'{stem}_completion.png'
    fig_path = SD_FIGURE_DIR / f'{stem}_comparison.png'

    gt.save(gt_path)
    masked.save(masked_path)
    mask.save(mask_path)
    result.save(result_path)

    metrics = compute_metrics(gt, result, mask)
    metrics.update({
        'index': i,
        'image_path': str(image_path),
        'ground_truth_path': str(gt_path),
        'masked_input_path': str(masked_path),
        'mask_path': str(mask_path),
        'completion_path': str(result_path),
        'seed': seed,
        'box_ratio': BOX_RATIO,
        'model': 'Stable Diffusion Inpainting',
    })
    records.append(metrics)

    show_comparison(gt, masked, mask, result, title=f'Stable Diffusion Inpainting Example {i}', save_path=fig_path)

metrics_df = pd.DataFrame(records)
metrics_csv = SD_METRIC_DIR / 'stable_diffusion_metrics.csv'
metrics_df.to_csv(metrics_csv, index=False)
print('Saved metrics to:', metrics_csv)
metrics_df.head()

## 8. Summarize Stable Diffusion metrics

These metrics provide a quantitative comparison point. However, generative inpainting may create visually realistic outputs that do not exactly match the original ground truth pixels. Therefore, the visual comparison is especially important.

In [ ]:
summary = metrics_df[[
    'l1_full', 'mse_full', 'psnr_full', 'ssim_full',
    'l1_masked_region', 'mse_masked_region'
]].agg(['mean', 'std']).T

summary_path = SD_METRIC_DIR / 'stable_diffusion_metric_summary.csv'
summary.to_csv(summary_path)
print('Saved metric summary to:', summary_path)
summary

## 9. Prompt and parameter tuning

Stable Diffusion inpainting is sensitive to the prompt and inference parameters. Use this section to test different settings on the same image and mask.

In [ ]:
tuning_prompts = [
    'a realistic photo of a dog, natural fur texture, realistic lighting, high quality',
    'a realistic close-up photo of a dog face with detailed fur and natural colors',
    'a high quality realistic dog photo, coherent fur texture, natural background',
]

tuning_results = []
for j, p in enumerate(tuning_prompts):
    result_j = run_sd_inpainting(
        image=masked,
        mask=mask,
        prompt=p,
        negative_prompt=negative_prompt,
        seed=SEED,
        guidance_scale=7.5,
        num_inference_steps=40
    )
    tuning_results.append(result_j)

fig, axes = plt.subplots(1, len(tuning_results) + 2, figsize=(5 * (len(tuning_results) + 2), 5))
axes[0].imshow(masked)
axes[0].set_title('Masked Input')
axes[0].axis('off')
axes[1].imshow(gt)
axes[1].set_title('Ground Truth')
axes[1].axis('off')

for j, result_j in enumerate(tuning_results):
    axes[j + 2].imshow(result_j)
    axes[j + 2].set_title(f'Prompt {j+1}')
    axes[j + 2].axis('off')

plt.tight_layout()
plt.savefig(SD_FIGURE_DIR / 'prompt_tuning_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Discussion

- CNN-based models are better for controlled quantitative reconstruction and might produce lower pixel by pixel restruction loss
- Stable Diffusion is better for realistic visual completion